# Laplace Correction - Starter Notebook
Demonstrates Laplace (add-one) smoothing in Naive Bayes to avoid zero-probability issues.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [ ]:
# Toy dataset: weather-based play decision
data = {
    'Outlook':    [0,0,1,2,2,2,1,0,0,2,0,1,1,2],
    'Temp':       [0,0,0,1,2,2,2,1,2,1,1,1,0,1],
    'Humidity':   [0,0,0,0,1,1,1,0,1,1,1,0,1,0],
    'Wind':       [0,1,0,0,0,1,1,0,0,0,1,1,0,1],
    'Play':       [0,0,1,1,1,0,1,0,1,1,1,1,1,0]
}
df = pd.DataFrame(data)
X = df.drop('Play', axis=1).values
y = df['Play'].values

In [ ]:
# Manual Naive Bayes without and with Laplace smoothing
def nb_predict(X_train, y_train, x_test, alpha=0):
    classes = np.unique(y_train)
    n_features = X_train.shape[1]
    log_probs = {}
    for c in classes:
        X_c = X_train[y_train == c]
        prior = np.log((len(X_c) + alpha) / (len(y_train) + alpha * len(classes)))
        log_lik = 0
        for j in range(n_features):
            n_vals = len(np.unique(X_train[:, j]))
            count = np.sum(X_c[:, j] == x_test[j])
            prob = (count + alpha) / (len(X_c) + alpha * n_vals)
            log_lik += np.log(prob + 1e-12 if alpha == 0 else prob)
        log_probs[c] = prior + log_lik
    return max(log_probs, key=log_probs.get)

# Test on full dataset
alphas = [0, 1]
for alpha in alphas:
    preds = [nb_predict(X, y, X[i], alpha=alpha) for i in range(len(X))]
    acc = accuracy_score(y, preds)
    label = 'No smoothing' if alpha == 0 else 'Laplace (alpha=1)'
    print(f"{label}: accuracy = {acc:.3f}")

In [ ]:
# Compare CategoricalNB with varying alpha
alphas = [0.001, 0.1, 0.5, 1.0, 2.0, 5.0]
accs = []
for a in alphas:
    clf = CategoricalNB(alpha=a)
    clf.fit(X, y)
    accs.append(accuracy_score(y, clf.predict(X)))

plt.figure(figsize=(7, 4))
plt.plot(alphas, accs, marker='o', color='steelblue')
plt.xscale('log')
plt.xlabel('Laplace alpha')
plt.ylabel('Train Accuracy')
plt.title('Effect of Laplace Correction on Naive Bayes')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Show zero-count problem and how alpha fixes it
print('--- Without Laplace (alpha~0): risk of log(0) ---')
clf0 = CategoricalNB(alpha=1e-10)
clf0.fit(X, y)
print('Log probs class 0:', clf0.feature_log_prob_[0])

print('\n--- With Laplace alpha=1 ---')
clf1 = CategoricalNB(alpha=1)
clf1.fit(X, y)
print('Log probs class 0:', clf1.feature_log_prob_[0])